# Harringtonine CropArray Example

---

## Notebook summary 


- Load a microscope image of video
- Tracking spots on the image and generate a pandas dataframe with the spots locations
- Creating a croparray with the image and dataframe
- Signal quantification and plotting
- Visualization of croparray with Napari

----

## Importing libraries


----

In [1]:
# To manipulate arrays
import numpy as np 
from skimage.io import imread
import matplotlib.pyplot as plt 
from matplotlib.path import Path
import pylab as pyl
import seaborn as sns; sns.set()  
import pathlib # for working with windows paths
import sys
import cv2
import os
import glob
import trackpy as tp
import pandas as pd
#!pip install shapely
from shapely.geometry import Polygon 
from shapely.geometry import Point
current_dir = pathlib.Path().absolute()
croparray_dir = current_dir.parents[0].joinpath('croparray')
sys.path.append(str(croparray_dir))
import crop_array_tools as ca
import os
import xarray as xr
# %matplotlib inline 
plt.style.use('dark_background')

# Napari 
%gui qt5 
import napari
from napari.utils import nbscreenshot

# Magicgui
from magicgui import magicgui
import datetime
import pathlib
import tifffile as tifffile

### Convert the Stacked movies from the Terminator to a Fiji hyperstack

In [2]:
# def convert_to_hyperstack(movie_data, num_channels, num_z_stacks, num_time_points):
#     # Create an empty hyperstack array with the correct shape
#     hyperstack_data = np.empty((num_time_points, num_z_stacks, num_channels, 512, 512), dtype=movie_data.dtype)

#     # Rearrange the data to the hyperstack format
#     for t in range(num_time_points):
#         for z in range(num_z_stacks):
#             for c in range(num_channels):
#                 frame_idx = t * (num_z_stacks * num_channels) + z * num_channels + c
#                 hyperstack_data[t, z, c] = movie_data[frame_idx]

#     return hyperstack_data

# def process_folder(folder_path):
#     # List all .tif files in the folder
#     tif_files = glob.glob(os.path.join(folder_path, '*.tif'))

#     # Load each .tif file, convert to hyperstack, and save the result in the parent directory with the folder name
#     for tif_file in tif_files:
#         print("Converting:", tif_file)
#         movie_data = tifffile.imread(tif_file)
#         hyperstack_data = convert_to_hyperstack(movie_data, num_channels, num_z_stacks, num_time_points)
        
#         # Get the folder name without path
#         folder_name = os.path.basename(folder_path)
        
#         # Save the hyperstack in the parent directory with the folder name and ".tif" extension
#         output_path = os.path.join(os.path.dirname(folder_path), folder_name + ".tif")
#         tifffile.imsave(output_path, hyperstack_data, imagej=True)
#         print("Hyperstack saved to:", output_path)

# # Assuming 'main_directory' is the path to the directory containing subdirectories with .tif files
# main_directory = r"Z:\koch\SARS-CoV-2 Project\Imaging\Terminator Imaging\Rapid\SARS_CTRL\20230725\150msExp_20T_0I\Dish02"
# num_channels = 2
# num_z_stacks = 13
# num_time_points = 20

# # Iterate over subdirectories and process each folder
# for root, dirs, files in os.walk(main_directory):
#     for folder in dirs:
#         folder_path = os.path.join(root, folder)
#         process_folder(folder_path)

## Functions 

___

In [3]:
def TrackWithMasks(img_max, particle_diameter ,min_m, mask_include, mask_exclude):
    f = tp.batch(img_max, diameter=particle_diameter,minmass=min_m)
    f_list = []
    for i in np.arange(len(f['frame'].unique())):
        f0 = f[f['frame']==i]
        f1 = f0.copy()

        # If no masks, include everything and exclude nothing
        if len(mask_include) == 0 :
            mask_include1 = [[0,0],[0,10000000],[10000000,10000000],[10000000,0]]
        else:
            mask_include1 = mask_include
        if len(mask_exclude) == 0:
            mask_exclude1 = [[0,0],[0,-10],[-10,-10]]
        else:
            mask_exclude1 = mask_exclude

        mask_in = Polygon(mask_include1)  # This is a polygon that defines the mask
        mypts = np.transpose([f1.y,f1.x]) # These are the points detected by trackpy, notice the x/y inversion for napari
        f1['Include']=[mask_in.contains(Point(mypts[i])) for i in np.arange(len(mypts))] # Check if pts are on/in polygon mask  

        # # Label points in nucleus if polygon mask exists
        # if polygon != None:
        mask_out = Polygon(mask_exclude1)  # This is a polygon that defines the mask
        f1['Exclude']=[mask_out.contains(Point(mypts[i])) for i in np.arange(len(mypts))] # Check if pts are on/in polygon mask  
        
        f_list.append(f1[(f1['Include']==True) & (f1['Exclude']==False)])
    f_all = pd.concat(f_list)
    return f_all

In [4]:
def TrackWithMasks1(img_max, particle_diameter, min_m, mask_include, mask_exclude, correction_factor, threshold_multiplier):
    f_list = []
    for i in np.arange(img_max.shape[0]):  # Loop through each timepoint
        threshold = threshold_multiplier * correction_factor[i, :, 1].mean()  # Channel 1 is translation

        # Detect particles using Trackpy with the dynamic threshold
        f0 = tp.locate(img_max[i], diameter=particle_diameter, minmass=min_m, threshold=threshold)

        if f0.empty:
            print(f"No particles found in frame {i}.")
            continue

        f1 = f0.copy()

        # Add 'frame' column manually
        f1['frame'] = i

        # If no masks, include everything and exclude nothing
        if len(mask_include) == 0:
            mask_include1 = [[0, 0], [0, 10000000], [10000000, 10000000], [10000000, 0]]
        else:
            mask_include1 = mask_include

        if len(mask_exclude) == 0:
            mask_exclude1 = [[0, 0], [0, -10], [-10, -10]]
        else:
            mask_exclude1 = mask_exclude

        # Create inclusion and exclusion masks
        mask_in = Polygon(mask_include1)
        mask_out = Polygon(mask_exclude1)
        mypts = np.transpose([f1.y, f1.x])  # Trackpy uses x/y, napari uses y/x
        
        # Filter by masks
        f1['Include'] = [mask_in.contains(Point(pt)) for pt in mypts]
        f1['Exclude'] = [mask_out.contains(Point(pt)) for pt in mypts]

        # Append to the list, filtering spots based on masks
        f_list.append(f1[(f1['Include'] == True) & (f1['Exclude'] == False)])

    # Concatenate all frames into a single DataFrame
    f_all = pd.concat(f_list)

    return f_all


In [5]:
def calculate_correction_factors(movie, N):
    """
    Calculate correction factors across all frames to match the total intensity
    of each z-plane to that of the specified frame N, using NumPy array operations.
   
    Parameters:
    - movie: numpy array with dimensions (t, z, x, y, ch),
             where t = timepoints, z = z-planes, x and y = spatial dimensions, ch = channels.
    - N: int, the frame index used as the reference for intensity matching.
   
    Returns:
    - correction_factors: numpy array with correction factors.
    """
    # Calculate the mean intensity for each z-plane and channel at each timepoint
    mean_intensities = movie.mean(axis=(2, 3))
   
    # Extract the mean intensity for the reference frame N for each z-plane and channel
    reference_intensities = mean_intensities[N]
   
    # Calculate the correction factors for each timepoint, z-plane, and channel
    correction_factors = np.ones_like(mean_intensities)  # Initialize to ones

    # Correct channel 0 using the reference intensity at N
    correction_factors[:, :, 0] = reference_intensities[:, 0] / mean_intensities[:, :, 0]

    # If there are more than 1 channel, correct channel 1 using the max mean value in the first 5 time points
    if movie.shape[-1] > 1:
        # Extract the mean intensity for the first 5 frames for each z-plane and channel 1
        mean_intensities_first_5 = mean_intensities[:5]
        # Find the maximum average value within the first 5 time points for each z-plane and channel 1
        max_mean_intensity_first_5_ch1 = mean_intensities_first_5[:, :, 1].max(axis=0)
        correction_factors[:, :, 1] = max_mean_intensity_first_5_ch1 / mean_intensities[:, :, 1]

    # If there are more than 2 channels, correct channel 2 using the reference intensity at N
    if movie.shape[-1] > 2:
        correction_factors[:, :, 2] = reference_intensities[:, 2] / mean_intensities[:, :, 2]
   
    # Avoid division by zero by setting any infinite correction factors to 1
    correction_factors[np.isinf(correction_factors)] = 1
   
    return correction_factors

In [6]:
import scipy.ndimage as ndi

def smooth_based_on_correction(movie, correction_factors):
    """
    Apply a Gaussian blur to each frame of the movie based on the correction factor.
   
    Parameters:
    - movie: numpy array with dimensions (t, z, x, y, ch),
             where t = timepoints, z = z-planes, x and y = spatial dimensions, ch = channels.
    - correction_factors: numpy array with dimensions (t, z, ch),
                          where t = timepoints, z = z-planes, ch = channels.
   
    Returns:
    - smoothed_movie: numpy array with smoothed frames.
    """
    # Initialize an array to hold the smoothed movie
    smoothed_movie = np.empty_like(movie)

    # Loop over each frame in the movie
    for t in range(movie.shape[0]):
        for z in range(movie.shape[1]):
            for ch in range(movie.shape[4]):
                # Calculate the sigma for the Gaussian blur based on the correction factor
                sigma = correction_factors[t, z, ch] / 3 # Divide by 3 for a smoother blur

                # Apply the Gaussian blur to the frame
                smoothed_movie[t, z, :, :, ch] = ndi.gaussian_filter(movie[t, z, :, :, ch], sigma=sigma)

    return smoothed_movie

## Calculate Correction Factor on Raw Movie

In [1308]:
cellname = 'Cell014'

In [1309]:
dir_raw = r"Z:\koch\SARS-CoV-2 Project\Imaging\Terminator Imaging\Rapid\Nsp1_Emi_NoStart\20260112\65msExp_20T_0I\Dish01"

In [1310]:
raw_img_filename = f"{cellname}.tif"
raw_img = imread(os.path.join(dir_raw,raw_img_filename))
raw_img.shape

(20, 13, 512, 512, 3)

In [1311]:
img_raw1 = raw_img.transpose([0,1,2,3,4])
print("Correction factor calculation format shape [f, z, y, x, ch] = ", img_raw1.shape)

Correction factor calculation format shape [f, z, y, x, ch] =  (20, 13, 512, 512, 3)


In [1312]:
correction_factors = calculate_correction_factors(img_raw1, 0)

## Loading in corrected transformed movie


----

In [1313]:
dir = r"Z:\koch\SARS-CoV-2 Project\Imaging\Terminator Imaging\Rapid\Nsp1_Emi_NoStart\20260112\65msExp_20T_0I\Dish01"
my_output_dir = dir

In [1314]:
#Video directory
img_4D_filename = f"{cellname}.tif"
img_4D =  imread(os.path.join(dir,img_4D_filename))
img_4D.shape

(20, 13, 512, 512, 3)

### Smooth movie based on correction factors

In [1315]:
img_4D = img_4D.transpose([0,1,2,3,4])
img_4D.shape
img_4D1 = smooth_based_on_correction(img_4D, correction_factors)
img_4D = img_4D1.transpose([0,1,2,3,4])
img_4D.shape

(20, 13, 512, 512, 3)

In [1316]:
# Converting the video to Croparray format
img_croparray = np.expand_dims(img_4D,axis=0) # 
img_croparray.shape # dimensions MUST be (fov, f , z, y, x, ch)
img_croparray.shape

#if you need to reorder dimensions
img_croparray1 = np.expand_dims(img_4D,axis=0) 
img_croparray = img_croparray1.transpose([0,1,2,3,4,5]) # dimensions MUST be (fov, f , z, y, x, ch)

print("croparray format shape [fov, f , z, y, x, ch] = ", img_croparray.shape)

croparray format shape [fov, f , z, y, x, ch] =  (1, 20, 13, 512, 512, 3)


### Translation Tracking

In [938]:
viewer1 = napari.view_image(img_croparray[0,:,:,:,:,1])

In [ ]:
best_zs = [0,6]
img_max = np.max(img_croparray[0,:,best_zs[0]:best_zs[1],:,:,1],axis=1)
img_max.shape

In [ ]:
viewer2 = napari.view_image(np.max(img_max,axis=0))  # Max of max, should be a single frame

In [ ]:
# First adjust the particle diameter and min mass to detect particles best in the first frame
# Then adjust the threshold multiplier to reduce the number of false positives in the later frames
particle_diameter = 5
min_mass  = 4000
threshold_multiplier = 175
viewer3 = napari.view_image(img_max)
f_all = TrackWithMasks1(img_max,particle_diameter,min_mass,viewer2.layers['Shapes [1]'].data[0][:,-2:],viewer2.layers['Shapes'].data[0][:,-2:],correction_factors,threshold_multiplier)
data = f_all[['frame','y','x']].values
layer_name = 'Spots all frames'
viewer3.add_points(data, size = 12, face_color='transparent', edge_color = 'yellow', symbol='disc', name = layer_name)

In [ ]:
f_all.to_csv(os.path.join(my_output_dir,img_4D_filename[:-4]+"_Translation"+'.csv'))

In [ ]:
#only if you didn't track:
spots = f_all.copy() # Nice to copy; seems it can cause to overwrite otherwise
spots['id']=spots.index
spots.rename(columns={'x': 'xc','y': 'yc', 'frame': 'f','signal':'signal_tp'}, 
inplace=True, errors='raise')
spots['fov']=0
spots.rename(columns={'particle':'id'})
spots = spots[['fov','id','f','yc','xc','signal_tp','Include','Exclude']] # keeping signal out of curiousity... want to compare to disk-donut measurements  
spots.head()

In [ ]:
my_ca = ca.create_crop_array(img_croparray,spots)#,xy_pad=5, dxy=130, dz=500, dt=1, units=['nm','min'], name = os.path.join(dir,img_4D_filename))
my_ca

In [ ]:
my_ca.to_netcdf(os.path.join(my_output_dir,img_4D_filename[:-4]+"_Translation"+'.nc') )

### mRNA Tracking

In [1317]:
viewer1 = napari.view_image(img_croparray[0,:,:,:,:,0])

In [1318]:
best_zs = [4,10]
img_max = np.max(img_croparray[0,:,best_zs[0]:best_zs[1],:,:,0],axis=1)
img_max.shape

(20, 512, 512)

Now you can create a mask to exclude regions: Shapes will be excluded and Shapes [1] will be included in tracks

In [1319]:
viewer2 = napari.view_image(np.max(img_max,axis=0))  # Max of max, should be a single frame

In [1322]:
# Use this if you don't have a mask:
# f_all = TrackWithMasks(img_max,7,1000,[],[])
particle_diameter = 5
min_mass = 2200
viewer3 = napari.view_image(img_max)
f_all = TrackWithMasks(img_max,particle_diameter,min_mass,viewer2.layers['Shapes [1]'].data[0][:,-2:],viewer2.layers['Shapes'].data[0][:,-2:])
data = f_all[['frame','y','x']].values
layer_name = 'Spots all frames'
viewer3.add_points(data, size = 12, face_color='transparent', edge_color = 'yellow', symbol='disc', name = layer_name)

Frame 19: 196 features


<Points layer 'Spots all frames' at 0x20d1506e220>

In [1323]:
f_all.to_csv(os.path.join(my_output_dir,img_4D_filename[:-4]+"_mRNA"+'.csv'))

Convert f to crop_array format

In [1324]:
#only if you didn't track:
spots = f_all.copy() # Nice to copy; seems it can cause to overwrite otherwise
spots['id']=spots.index
spots.rename(columns={'x': 'xc','y': 'yc', 'frame': 'f','signal':'signal_tp'}, 
inplace=True, errors='raise')
spots['fov']=0
spots.rename(columns={'particle':'id'})
spots = spots[['fov','id','f','yc','xc','signal_tp','Include','Exclude']] # keeping signal out of curiousity... want to compare to disk-donut measurements  
spots.head()

,fov,id,f,yc,xc,signal_tp,Include,Exclude
3,0,3,0,146.773271,64.826304,740.996188,True,False
4,0,4,0,153.979461,107.014811,285.029683,True,False
5,0,5,0,157.944776,122.775906,530.020567,True,False
6,0,6,0,175.281912,82.041127,763.217469,True,False
7,0,7,0,177.324152,107.875887,368.397449,True,False


In [1325]:
my_ca = ca.create_crop_array(img_croparray,spots)#,xy_pad=5, dxy=130, dz=500, dt=1, units=['nm','min'], name = os.path.join(dir,img_4D_filename))
my_ca

Original video dimensions:  (1, 20, 13, 512, 512, 3)
Padded video dimensions:  (1, 20, 13, 524, 524, 3)
Max # of spots per frame:  226
Shape of numpy array to hold all crop intensity data:  (1, 226, 20, 13, 11, 11, 3)
Shape of xc and yc numpy arrays:  (1, 226, 20, 3)
Shape of extra my_layers numpy array:  (4, 1, 226, 20)


<xarray.Dataset>
Dimensions:    (fov: 1, n: 226, t: 20, z: 13, y: 11, x: 11, ch: 3)
Coordinates:
  * fov        (fov) int32 0
  * n          (n) int16 0 1 2 3 4 5 6 7 8 ... 218 219 220 221 222 223 224 225
  * t          (t) int32 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
  * z          (z) int32 0 1 2 3 4 5 6 7 8 9 10 11 12
  * y          (y) int32 -5 -4 -3 -2 -1 0 1 2 3 4 5
  * x          (x) int32 -5 -4 -3 -2 -1 0 1 2 3 4 5
  * ch         (ch) int32 0 1 2
Data variables:
    int        (fov, n, t, z, y, x, ch) int32 7503 6975 5291 6823 ... 0 0 0 0
    xc         (fov, n, t, ch) int32 71 71 71 73 73 73 69 69 ... 0 0 0 0 0 0 0 0
    yc         (fov, n, t, ch) int32 153 153 153 155 155 155 155 ... 0 0 0 0 0 0
    dx         int32 1
    dy         int32 1
    dz         int32 1
    dt         int32 1
    xy_pad     int32 5
    id         (fov, n, t) float64 3.0 230.0 450.0 669.0 ... 0.0 0.0 0.0 0.0
    signal_tp  (fov, n, t) float64 741.0 586.0 800.0 619.0 ... 0.0 0.0 0.0 0.0
    Include    (fov, n, t) float64 1.0 1.0 1.0 1.0 1.0 ... 0.0 0.0 0.0 0.0 0.0
    Exclude    (fov, n, t) float64 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Attributes:
    name:     video_filename
    date:     video_date

In [ ]:
my_ca.to_netcdf(os.path.join(my_output_dir,img_4D_filename[:-4]+"_mRNA"+'.nc') )

: 

## Quantify signal intensity through time
____

In [ ]:
#OPEN an .NC file
dir=r'\\129.82.125.210\TSnas2\galindo\1_Imaging_Data\Madesh_UT\DataAnalysis\G3P\5mM'
my_ca=xr.open_dataset(os.path.join(dir,'Cell03.nc'))

In [ ]:
# Measure signals and plot average signal through time, creating 'best_z' layer and 'signal' layer
ca.measure_signal(my_ca, ref_ch=0, disk_r=3, roll_n=3)

In [ ]:
my_ca.best_z.mean('n').sel(fov=0,ch=0).rolling(t=3,min_periods=1).mean().plot.imshow(col='t',col_wrap=10,robust=True,xticks=[],yticks=[],size=1.5,cmap='gray', vmin=0, vmax =500)

Let's compare our disk-donut 'signal' layer (acquired from 3D image) to trackpy's (acquired from max-projection):

In [ ]:
# Let's compare our intensity numbers to those from trackpy:
my_ca.where(my_ca.signal>0).plot.scatter(x='signal',y='signal_tp',col='ch',hue='ch',colors=['red','limegreen','blue'],levels=[0,1,2,3])

Let's look at average signal vs time

In [ ]:
# Let's look at average signal vs time
start_sig = my_ca.signal.mean('n').sel(t=slice(0,0)).mean('t')
end_sig = 0# my_ca.signal.mean('n').sel(t=slice(15,20)).mean('t')
norm_sig = (my_ca.signal.mean('n') - end_sig)/(start_sig - end_sig)
sns.set_palette(['limegreen','limegreen','blue'])
norm_sig.sel(fov=0,ch=0).plot.line(x='t',hue='ch')#,ylim=[0,1.5])

Now let's just use trackpy's values:

In [ ]:
# Let's look at average signal vs time
start_sig = my_ca.signal_tp.mean('n').sel(t=slice(0,4)).mean('t')
end_sig = 0# my_ca.signal_tp.mean('n').sel(t=slice(15,20)).mean('t')
norm_sig = (my_ca.signal_tp.mean('n') - end_sig)/(start_sig - end_sig)
sns.set_palette(['limegreen','limegreen','blue'])
norm_sig.sel(fov=0).plot.line(x='t',hue='ch')

I guess trackpy and the disk donut method do a very good job at getting the intensities of spots. Although note that trackpy got the values from the max-intensity projection. Interesting.

## Visualize crop array montage with Napari
___

Now let's see a montage of the selected spots' best-z planes:

In [ ]:
# view the action of montage showing an n x n crop_array through time
viewer = napari.view_image(ca.montage(my_ca.sel(fov=0,ch=0).best_z,row='n',col='t'),contrast_limits=[60,800])

## Optional: Create Track Array
___

In [ ]:
f_all

In [ ]:
# Note, if you actually wanted to track, you could use the following:
# link tracks
max_distance_movement = 15
track_skip_frames = 3
min_trajectory_length = 10
t = tp.link(f_all, max_distance_movement, memory=track_skip_frames)
t1 = tp.filter_stubs(t, min_trajectory_length)
t1['particle'] = t1['particle']+1 # VERY IMPORTANT NOT TO HAVE TRACK IDs WITH VALUES = 0 WHEN MAKING CROP ARRAYS AS ZERO IS DEFAULT EMPTY VALUE
# Compare the number of particles in the unfiltered and filtered data.
print('Before:', t['particle'].nunique())
print('After:', t1['particle'].nunique()) 

In [ ]:
# only if you tracked:
spots = t1.copy()
spots.rename(columns={'x': 'xc','y': 'yc', 'frame': 'f','signal':'signal_tp','particle':'id'}, 
inplace=True, errors='raise')


spots['fov']=0
spots.rename(columns={'particle':'id'})
spots = spots[['fov','id','f','yc','xc','signal_tp']] # keeping signal out of curiousity... want to compare to disk-donut measurements  
spots.head()

In [ ]:
my_ca2 = ca.create_crop_array(img_croparray,spots,xy_pad=particle_diameter//2+1, dxy=130, dz=500, dt=1, units=['nm','min'])

In [ ]:
# Measure signals and plot average signal through time, creating 'best_z' layer and 'signal' layer
ca.measure_signal(my_ca2, ref_ch=1, disk_r=3, roll_n=3)

In [ ]:
import xarray as xr
import pandas as pd
# Since ids correspond to tracks, we can organize tracks in rows  
my_ids = np.unique(my_ca2.id) # Find all unique ids
my_ids = my_ids[1:] # remove the '0' ID used as filler in Crop Arrays
my_ids
# Get a list of xarrays for each unique id 
my_das = []
for i in np.arange(len(my_ids)):
    temp = my_ca2.groupby('id')[my_ids[i]].reset_index('stacked_fov_n_t').reset_coords('n',drop=True).reset_coords('fov',drop=True).swap_dims({'stacked_fov_n_t':'t'})
    my_das.append(temp)
    del temp
# Concatenate the xarrays together to make a new xarray dataset in a track array format (each track on separate row). Here 'n' is replaced by 'tracks'
my_taz = xr.concat(my_das, dim=pd.Index(my_ids, name='track_id'), fill_value=255) # fill_value=0 so keep int instead of moving to floats with NaNs
my_taz = my_taz.transpose('track_id','fov','n','t','z','y','x','ch', missing_dims='ignore') # reorder for napari
my_taz

In [ ]:
# view the action of montage showing an n x n crop_array through time
viewer = napari.view_image(ca.montage(my_taz.sel(ch=1).best_z,row='track_id',col='t'),contrast_limits=[60,800])

In [ ]:
my_taz.isel(track_id=[12,16,20]).sel(ch=1).signal.plot.line(x='t',col='track_id',col_wrap=3)

## Analyzing groups of crop arrays


----

In [ ]:
# Data filename and directory
subdir_name = '5mM'
dir_1 = r'\\129.82.125.210\TSnas2\galindo\1_Imaging_Data\Madesh_UT\DataAnalysis\G3P'
dir = os.path.join(dir_1,subdir_name)
dir

In [ ]:
import glob

my_files = glob.glob(os.path.join(dir,'*.nc'))
my_files

In [ ]:
import xarray as xr
my_cas = [xr.open_dataset(files) for files in my_files]
for i in my_cas:
    ca.measure_signal(i, ref_ch=1, disk_r=4, roll_n=3)

In [ ]:
my_ca_tot = xr.concat(my_cas, dim="n")

In [ ]:
os.path.join(dir_1,subdir_name+'.svg')

In [ ]:
plt.style.use('seaborn-whitegrid')
sns.set_palette(['green'])
my_data = my_ca_tot.signal.sel(fov=0,ch=1).to_pandas()
my_data_df = pd.melt(my_data,value_name="Translation Intensity (arb.u.)").rename(columns={'t':'Time (min)'})
g=sns.relplot(x='Time (min)', y='Translation Intensity (arb.u.)',data=my_data_df,kind='line',height=3,aspect=1)
plt.savefig(os.path.join(dir,subdir_name+'.svg'))

In [ ]:
# Now that we've organized things by id (track), we can easily normalize track data
start_sig = my_ca_tot.signal.sel(t=slice(0,5)).mean(['n','t'])
norm_sig = my_ca_tot.signal/start_sig

In [ ]:
plt.style.use('seaborn-whitegrid')
sns.set_palette(['green'])
my_data = norm_sig.sel(fov=0,ch=1).to_pandas()
my_data_df = pd.melt(my_data,value_name="Translation Intensity (arb.u.)").rename(columns={'t':'Time (min)'})
g=sns.relplot(x='Time (min)', y='Translation Intensity (arb.u.)',data=my_data_df,kind='line',height=3,aspect=1,)
plt.ylim(0.1,1.7)
my_data_df[((my_data_df['Time (min)']>40) & (my_data_df['Time (min)']<51))].to_csv(os.path.join(dir,subdir_name+'_normalized.csv')) 
plt.savefig(os.path.join(dir,subdir_name+'_normalized.svg'))

In [ ]:
# Read in csv files for comparison
my_dirs = ['Lactate','Pep','KoningicAcid','G3P','DHAP','CitricAcid','CHX_100ugmL','Vehicle','CHX_Puro']
my_csvs = [os.path.join(dir_1,os.path.join(i,i+'_normalized.csv')) for i in my_dirs]
my_dfs = [pd.read_csv(i) for i in my_csvs]

In [ ]:
ti_list = []
for i in np.arange(len(my_dirs)):
    temp = my_dfs[i]['Translation Intensity (arb.u.)']
    temp2 = [x for x in temp if 0<x<10000]
    ti_list.append(temp2)

In [ ]:
my_df_all = [pd.DataFrame({'Relative Change in Translation Signal':ti_list[i]}) for i in np.arange(len(my_dirs))]
for i in np.arange(len(my_dirs)):
    my_df_all[i]['Molecule'] = my_dirs[i]
my_df_all_vert = pd.concat(my_df_all)

In [ ]:
temp = my_df_all_vert
sns.boxplot(data = temp,x='Molecule',y='Relative Change in Translation Signal',showfliers=False)
plt.ylim(0,6.5)



# Trying to track in x, y, and z:

Measure signals and plot average signal through time, creating 'best_z' layer and 'signal' layer

In [ ]:
max_distance_movement = 5
track_skip_frames = 3
my_frame = 15
min_trajectory_length=2
my_list = []
for my_frame in np.arange(len(img_croparray[0,:])): #np.arange(15,16,1):
    f = tp.batch(img_croparray[0,my_frame,:,:,:,1], diameter=7,minmass=1000)
    t1 = tp.link(f, max_distance_movement, memory=track_skip_frames)
    t = tp.filter_stubs(t1, min_trajectory_length)
    sort_t = t.sort_values(['particle']).groupby('particle').aggregate('max').reset_index().rename(columns={'frame':'z'})
    sort_t['t']=my_frame
    my_list.append(sort_t)
my_df = pd.concat(my_list)
my_df.reset_index()

In [ ]:
import pandas as pd
my_df = pd.concat(my_list)
my_df.reset_index()

In [ ]:
my_df['raw_mass'].hist()

In [ ]:
plt.figure(figsize=(20,10))
tp.annotate(my_df[(my_df['t']==28)&(my_df['raw_mass']<40000)], img_4D_max_real[28]);

In [ ]:
np.array([len(t[t['particle']==i].x) for i in np.arange(len(t.particle))])

In [ ]:
f[f['frame']==6]

# GUI 
___

In [ ]:
def my_transpose_list(ar, input_list):
    my_length = len(input_list)  # Should always be 6 = ['fov','f','z','y','x','ch']
    my_none_count = input_list.count('None')  # Get number of 'None' elements in current image
    my_expand_dim_list = tuple(-(np.arange(my_none_count)+1))
    my_count = len(input_list) - my_none_count
    output_list = []
    counter = my_count
    if my_count > 0: 
        for i in np.arange(my_length):
            if input_list[i] == 'None':
                output_list.append(counter)
                counter = counter + 1
            else:
                output_list.append(input_list[i])
    output_ar = np.expand_dims(ar,my_expand_dim_list)
    return np.transpose(output_ar, output_list)
input_list = [0,'None',3,2,'None',1]
my_croparray = my_transpose_list(np.zeros([10,11,12,13]),input_list)
my_croparray.shape

In [ ]:
my_none_count = 3
my_test_array = np.zeros([1,2,3])
my_test_array.shape
np.expand_dims(np.zeros([1,2,3]),my_test_array,axis=(-1,-2,-3)).shape


In [ ]:
from napari.layers import Image, Shapes
from napari.types import LabelsData, ImageData, ShapesData
import napari.types
@magicgui(
    call_button = 'Define Image Dimensions',
    fov_axis = {'choices':['None',0, 1, 2, 3, 4, 5]},
    f_axis = {'choices':['None',0, 1, 2, 3, 4, 5]},
    z_axis = {'choices':['None',0, 1, 2, 3, 4, 5]},
    y_axis = {'choices':['None',0, 1, 2, 3, 4, 5]},
    x_axis = {'choices':['None',0, 1, 2, 3, 4, 5]},
    ch_axis = {'choices':['None',0, 1, 2, 3, 4, 5]},            
)
def define_image_dims(
    image: Image,
    fov_axis: 'None',
    f_axis: 'None',
    z_axis: 'None',
    y_axis: 'None',
    x_axis: 'None',
    ch_axis: 'None',
):
    my_croparray = my_transpose_list(image.data,[fov_axis,f_axis,z_axis,y_axis,x_axis,ch_axis])
    print(my_croparray.shape)

viewer = napari.Viewer()
dw1 = viewer.window.add_dock_widget(define_image_dims, name='Get Video',area='right')

In [ ]:

#viewer.dims.current_step[0] This is how you can access the slider slice in napari

# @magicgui(call_button = 'Load File')
# def open_file(
#     number_of_channels: int,
#     filename = pathlib.Path('/some/path.ext')
# ) -> napari.types.LayerDataTuple:
#     img = imread(filename)
#     my_channel_axis = np.where(np.array(img.shape) == number_of_channels)[0][0]
#     img = np.moveaxis(img_4D,my_channel_axis,0)
#     return [(img[i],{'colormap':'gray','name':'Channel'+str(i)}) for i in np.arange(len(img))]
@magicgui(call_button = 'Make Tracking Channel Layer')
def get_channel(
    image: Image,
    channel_axis: int,
    channel_num: int,
) -> napari.types.LayerDataTuple:
    return (np.expand_dims(np.moveaxis(image.data,channel_axis,0)[channel_num],axis=channel_axis),{'name':image.name+'Ch '+str(channel_num)})

@magicgui(call_button = 'Make Max Projection')
def max_project(
    image1: Image,
    projection_axis: int,
    start_slice: int,
    stop_slice: int,
) -> napari.types.LayerDataTuple:
    return (np.expand_dims(np.max(image1.data, axis=projection_axis),axis=projection_axis),{'name':'Max of '+image1.name})

@magicgui(call_button = 'Detect Spots in Max Projection')
def detect_spots(
    image: Image,
    mask_include: Shapes,
    mask_exclude: Shapes,
    test_frame: int,
    size = 9,
    min_m = 1000,
    detect_spots_in_all_frames = False,
) -> napari.types.LayerDataTuple:
    if detect_spots_in_all_frames:
        f_all = TrackWithMasks(np.squeeze(image.data),size,min_m,mask_include.data[0][:,-2:],mask_exclude.data[0][:,-2:])
        print(f_all)
        print(np.squeeze(image.data).shape)
        data = f_all[['frame','y','x']].values
        data['ch']=1
        layer_name = 'Spots all frames'
    else:
        f = tp.locate(np.squeeze(image.data[test_frame]),diameter=size,minmass=min_m)
        print(f)
        data = f[['y','x']].values
        layer_name = 'Spots frame '+str(test_frame)
    #save_croparray.dir.value = 'test'
    #save_croparray.show()
    return [(data,{'size':size,'edge_color':'yellow','opacity':0.4,'symbol':'ring','name':layer_name}, 'points'),
            (np.max(image,axis=0), {'name': 'Max Projection', 'blending': 'additive'})]

viewer = napari.Viewer()
#viewer = napari.view_image(img_max, name="My Image")
#viewer.window.add_dock_widget(detect_spots)
#viewer.window.add_dock_widget(max_project)
#viewer.window.add_dock_widget(open_file, name='Step 1',area='right')
viewer.window.add_dock_widget(get_channel, name='Get Video',area='right')
dw2 = viewer.window.add_dock_widget(max_project, name='Max Project',area='right')
dw3 = viewer.window.add_dock_widget(detect_spots, name ='Tracking',area='right')
#viewer.window._qt_window.tabifyDockWidget(dw2, dw3)
#viewer.layers.events.changed.connect(detect_spots.reset_choices)

## Cellpose Tests

In [ ]:
dir = r'\\129.82.125.210\TSnas2\galindo\1_Imaging_Data\Madesh_UT\20220210_metabolites\PEP_10mM'

In [ ]:
#Video directory
img_4D_filename = 'Cell02.tif'
img_4D =  imread(os.path.join(dir,img_4D_filename))
img_4D.shape

In [ ]:
# Converting the video to Croparray format
img_croparray = np.expand_dims(img_4D,axis=0) # 
img_croparray.shape # dimensions MUST be (fov, f , z, y, x, ch)
img_croparray.shape

#if you need to reorder dimensions
img_croparray1 = np.expand_dims(img_4D,axis=0) 
img_croparray = img_croparray1.transpose([0,1,2,3,4,5]) # dimensions MUST be (fov, f , z, y, x, ch)

print("croparray format shape [fov, f , z, y, x, ch] = ", img_croparray.shape)

In [ ]:
from cellpose import models
from cellpose import plot

plt.imshow(img_4D[0,7,:,:,1])



In [ ]:
best_zs = [3,10]
img_max = np.max(img_croparray[0,:,best_zs[0]:best_zs[1],:,:,1],axis=1)
img_max.shape
plt.imshow(img_max[0,:,:])

In [ ]:
use_GPU = True
model = models.Cellpose(gpu=use_GPU, model_type='cyto') # model_type='cyto' or model_type='nuclei'
masks, flows, styles, diams = model.eval(img_max[0,:,:], diameter=200, flow_threshold=None, channels=[0,0],omni=True)
plt.imshow(masks,cmap='Greys')
plt.show()

In [ ]:
use_GPU = True
model = models.Cellpose(gpu=use_GPU, model_type='nuclei') # model_type='cyto' or model_type='nuclei'
masks, flows, styles, diams = model.eval(img_max[0,:,:], diameter=100, flow_threshold=None, channels=[0,0],omni=True)
plt.imshow(masks,cmap='Greys')
plt.show()

In [ ]:
model = models.Cellpose(gpu=use_GPU, model_type='cyto') # model_type='cyto' or model_type='nuclei'

# Running the model for all  frames
masks_for_all_frames, _, _, _ = model.eval(imgs_2[:,:,:,:], diameter=200, min_size=1000, channels=[0,0])

# Notice that the output is a tensor with  shape [T,Y,X]
print(masks_for_all_frames.shape)

# Plotting the  masks founds for every frame.
number_frames= imgs_2.shape[0]
fig, ax = plt.subplots(nrows=1,ncols=number_frames, figsize=(20, 5))
for i in range(number_frames):
  ax[i].imshow(masks_for_all_frames[i,:,:])
  ax[i].axis('off')